In [1]:
import numpy as np
import pandas as pd
import h5py
import os
from scipy.stats import skew, kurtosis
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [2]:
path = '/home/sz4544/EEG-motor-imagery-main/project/'

In [3]:
data_dict = h5py.File(os.path.join(path, "raw_EEG.h5"), "r")
data = data_dict['data'][:]
tasks = data_dict['tasks'][:]
reps = data_dict['reps'][:]
trials = data_dict['trials'][:]
trial_reps = data_dict['trial_reps'][:]
subjects = data_dict['subjects'][:]

In [4]:
# train-valid-test split: # about 72-14-14
np.random.seed(42)
which_trial_reps = np.unique(trial_reps)
tr_trial_reps = np.random.choice(which_trial_reps, 5, replace=False)
val_trial_reps = 4
ts_trial_reps = 7

tr_ind = np.where((trial_reps == 1) | (trial_reps == 2) | (trial_reps == 3) | (trial_reps == 5) | (trial_reps == 6))[0]
val_ind = np.where(trial_reps == val_trial_reps)[0]
ts_ind = np.where(trial_reps == ts_trial_reps)[0]

tr_data, tr_tasks = data[tr_ind], tasks[tr_ind]
val_data, val_tasks = data[val_ind], tasks[val_ind]
ts_data, ts_tasks = data[ts_ind], tasks[ts_ind]

In [5]:
Xtr, ytr = tr_data[np.where(subjects[tr_ind] <= 15)[0]], tr_tasks[np.where(subjects[tr_ind] <= 15)[0]]
Xval, yval = val_data[np.where(subjects[val_ind] <= 15)[0]], val_tasks[np.where(subjects[val_ind] <= 15)[0]]
Xts, yts = ts_data[np.where(subjects[ts_ind] <= 15)[0]], ts_tasks[np.where(subjects[ts_ind] <= 15)[0]]

tr_subjects = subjects[tr_ind][np.where(subjects[tr_ind] <= 15)[0]]
val_subjects = subjects[val_ind][np.where(subjects[val_ind] <= 15)[0]]
ts_subjects = subjects[ts_ind][np.where(subjects[ts_ind] <= 15)[0]]

In [6]:
print(Xtr.shape, ' ', ytr.shape)
print(Xval.shape, ' ', yval.shape)
print(Xts.shape, ' ', yts.shape)

(1800, 640, 64)   (1800,)
(360, 640, 64)   (360,)
(360, 640, 64)   (360,)


In [7]:
# ===== create sliding-window TRAIN + sliding-window VALID/TEST for trial vote =====
window_len = 512
stride = 128

def make_windows_with_trial_ids(X, y, subjects, window_len=512, stride=128):
    Xw, yw, sw, tw = [], [], [], []
    n_trials, trial_len, n_ch = X.shape

    for i in range(n_trials):
        for start in range(0, trial_len - window_len + 1, stride):
            end = start + window_len
            Xw.append(X[i, start:end, :])
            yw.append(y[i])
            sw.append(subjects[i])
            tw.append(i)

    Xw = np.stack(Xw)
    yw = np.array(yw)
    sw = np.array(sw)
    tw = np.array(tw)
    return Xw, yw, sw, tw

# train: sliding windows
Xtr_w, ytr_w, tr_subjects_w, tr_trial_ids_w = make_windows_with_trial_ids(
    Xtr, ytr, tr_subjects,
    window_len=window_len,
    stride=stride
)

# valid: sliding windows
Xval_w, yval_w, val_subjects_w, val_trial_ids_w = make_windows_with_trial_ids(
    Xval, yval, val_subjects,
    window_len=window_len,
    stride=stride
)

# test: sliding windows
Xts_w, yts_w, ts_subjects_w, ts_trial_ids_w = make_windows_with_trial_ids(
    Xts, yts, ts_subjects,
    window_len=window_len,
    stride=stride
)

print("original train:", Xtr.shape, ytr.shape)
print("windowed train:", Xtr_w.shape, ytr_w.shape)
print("windowed train labels:", np.unique(ytr_w, return_counts=True))

print("windowed valid:", Xval_w.shape, yval_w.shape)
print("windowed valid unique trials:", len(np.unique(val_trial_ids_w)))

print("windowed test:", Xts_w.shape, yts_w.shape)
print("windowed test unique trials:", len(np.unique(ts_trial_ids_w)))

# save files
f_train = h5py.File(os.path.join(path, 'train1800_window512s128_raw_EEG.h5'), 'w')
f_train.create_dataset('data', data=Xtr_w)
f_train.create_dataset('tasks', data=ytr_w)
f_train.create_dataset('subjects', data=tr_subjects_w)
f_train.create_dataset('trial_ids', data=tr_trial_ids_w)

f_valid = h5py.File(os.path.join(path, 'valid360_window512s128_trialvote_raw_EEG.h5'), 'w')
f_valid.create_dataset('data', data=Xval_w)
f_valid.create_dataset('tasks', data=yval_w)
f_valid.create_dataset('subjects', data=val_subjects_w)
f_valid.create_dataset('trial_ids', data=val_trial_ids_w)

f_test = h5py.File(os.path.join(path, 'test360_window512s128_trialvote_raw_EEG.h5'), 'w')
f_test.create_dataset('data', data=Xts_w)
f_test.create_dataset('tasks', data=yts_w)
f_test.create_dataset('subjects', data=ts_subjects_w)
f_test.create_dataset('trial_ids', data=ts_trial_ids_w)

f_train.close()
f_valid.close()
f_test.close()

print("saved:",
      os.path.join(path, 'train1800_window512s128_raw_EEG.h5'),
      os.path.join(path, 'valid360_window512s128_trialvote_raw_EEG.h5'),
      os.path.join(path, 'test360_window512s128_trialvote_raw_EEG.h5'))

original train: (1800, 640, 64) (1800,)
windowed train: (3600, 512, 64) (3600,)
windowed train labels: (array([1., 2., 3., 4.]), array([900, 900, 900, 900]))
windowed valid: (720, 512, 64) (720,)
windowed valid unique trials: 360
windowed test: (720, 512, 64) (720,)
windowed test unique trials: 360
saved: /home/sz4544/EEG-motor-imagery-main/project/train1800_window512s128_raw_EEG.h5 /home/sz4544/EEG-motor-imagery-main/project/valid360_window512s128_trialvote_raw_EEG.h5 /home/sz4544/EEG-motor-imagery-main/project/test360_window512s128_trialvote_raw_EEG.h5
